# NISAR GCOV — End-to-End Capstone


## Geographic checkpoint

Same workflow, with spatial context carried through end to end.

# 11 — End-to-End Capstone

Run the whole pipeline start to finish, ideally on a product you haven't already worked with in the earlier modules.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


In [ ]:
from nisar_utils.gcov import open_gcov, get_grid_coordinates, read_window
from nisar_utils.spatial import coordinate_to_index
from nisar_utils.statistics import summary, robust_limits

grid=f"{profile.gcov_root}/grids/{freq}"
term=(diagonal_terms or terms)[0]
with open_gcov(NISAR_FILE) as f:
    x,y=get_grid_coordinates(f,grid)
    (r0,r1,c0,c1),aoi=resolve_window(x,y,cfg,coordinate_to_index_func=coordinate_to_index)
    arr=read_window(f,f"{grid}/{term}",r0,r1,c0,c1)

print("CAPSTONE")
print("Source:",NISAR_FILE)
print("Profile:",profile.sar_family,profile.band,profile.product_level,profile.product_type)
print("GCOV root:",profile.gcov_root)
print("Frequency:",freq)
print("Term:",term)
print("Window:",arr.shape)
print("AOI configured:",bool(aoi))
print("Statistics:",summary(arr))
print("Robust limits:",robust_limits(arr))
print("\nModule 11 STATUS: PASS")


In [ ]:
from IPython.display import display

try:
    from nisar_utils.gcov import open_gcov, get_grid_coordinates
    from nisar_utils.mapping import folium_scene_map

    _grid = f"{profile.gcov_root}/grids/{freq}"

    with open_gcov(NISAR_FILE) as _f:
        _x, _y = get_grid_coordinates(_f, _grid)

    print("Grid:", _grid)
    print("X coordinates:", len(_x))
    print("Y coordinates:", len(_y))
    print("EPSG:", profile.epsg)

    # ------------------------------------------------------------
    # Normalize the geographic AOI saved by Module 06
    # ------------------------------------------------------------
    _aoi11 = cfg.get("default_aoi") or cfg.get("aoi")

    print("Stored AOI:", _aoi11)

    if _aoi11 and all(k in _aoi11 for k in
                      ("xmin", "xmax", "ymin", "ymax")):

        _map_aoi11 = {
            "lon_min": float(_aoi11["xmin"]),
            "lon_max": float(_aoi11["xmax"]),
            "lat_min": float(_aoi11["ymin"]),
            "lat_max": float(_aoi11["ymax"]),
        }

    elif _aoi11 and all(k in _aoi11 for k in
                        ("lon_min", "lon_max", "lat_min", "lat_max")):

        _map_aoi11 = {
            "lon_min": float(_aoi11["lon_min"]),
            "lon_max": float(_aoi11["lon_max"]),
            "lat_min": float(_aoi11["lat_min"]),
            "lat_max": float(_aoi11["lat_max"]),
        }

    else:
        _map_aoi11 = None

    print("Mapping AOI:", _map_aoi11)

    # ------------------------------------------------------------
    # Create GIS checkpoint map
    # ------------------------------------------------------------
    _m = folium_scene_map(
        _x,
        _y,
        profile.epsg,
        title="NISAR GIS checkpoint",
        aoi=_map_aoi11
    )

    print("Map object created:", type(_m))

    # Explicit Jupyter rendering
    display(_m)

except Exception as e:
    print("Spatial checkpoint unavailable:")
    print(type(e).__name__, ":", e)